In [1]:
import tensorflow as tf
import pandas as pd
import os

tf.config.run_functions_eagerly(True)

2024-09-14 01:11:17.026319: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-09-14 01:11:17.051334: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-09-14 01:11:17.051359: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-14 01:11:17.051373: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-09-14 01:11:17.059075: I tensorflow/core/platform/cpu_feature_g

In [2]:
train_folders = []
val_folders = []

for folder in os.listdir('training'):
    base_folder = os.path.join('training', folder)
    train_folders.append(os.path.join(base_folder, 'train.csv'))
    val_folders.append(os.path.join(base_folder, 'val.csv'))

In [3]:
train_files = tf.data.Dataset.list_files(train_folders)
val_files = tf.data.Dataset.list_files(val_folders)

2024-09-14 01:11:19.150062: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-09-14 01:11:19.159538: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-09-14 01:11:19.159567: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-09-14 01:11:19.162321: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-09-14 01:11:19.162341: I tensorflow/compile

In [4]:
def preprocess_data(features, label):
    features = tf.stack([
        features['open_normalized'], 
        features['high_normalized'], 
        features['low_normalized'], 
        features['close_normalized']], axis=1)
    
    return features, label


In [7]:
def create_time_series_window(features, labels, sequence_length=20):
    features = tf.convert_to_tensor(features, dtype=tf.float32)
    labels = tf.convert_to_tensor(labels, dtype=tf.int64)

    # Create sequences
    windows = []
    window_labels = []
    
    for i in range(len(features) - sequence_length):
        window = features[i:i + sequence_length]
        label = labels[i + sequence_length]
        windows.append(window)
        window_labels.append(label)
    
    return tf.data.Dataset.from_tensor_slices((tf.stack(windows), tf.stack(window_labels)))


In [11]:
@tf.function()
def preprocess_dataset(file_list, sequence_length=20):
    def parse_csv(record):
        columns = ['open_normalized', 'high_normalized', 'low_normalized', 'close_normalized', 'target']
        features = {col: record[col] for col in columns}
        labels = record['target']
        features = tf.stack([features['open_normalized'], features['high_normalized'], features['low_normalized'], features['close_normalized']], axis=1)
        return features, labels

    dataset = tf.data.Dataset.from_tensor_slices(file_list)
    dataset = dataset.interleave(lambda file: tf.data.experimental.make_csv_dataset(
        file,
        batch_size=10000,  # Batch size large enough to include the full dataset
        column_names=['open_normalized', 'high_normalized', 'low_normalized', 'close_normalized', 'target'],
        label_name='target',
        num_epochs=1,
        header=True
    ), cycle_length=1, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.map(parse_csv)

    # Convert to a format suitable for time series
    dataset = dataset.batch(1000)  # Adjust batch size if needed
    dataset = dataset.flat_map(lambda features, labels: create_time_series_window(features, labels, sequence_length))
    
    return dataset


In [13]:
train_dataset = preprocess_dataset(train_folders)
val_dataset = preprocess_dataset(val_folders)

OperatorNotAllowedInGraphError: in user code:

    File "/tmp/ipykernel_55262/2013467955.py", line 17, in None  *
        num_epochs=1,

    OperatorNotAllowedInGraphError: Iterating over a symbolic `tf.Tensor` is not allowed. You can attempt the following resolutions to the problem: If you are running in Graph mode, use Eager execution mode or decorate this function with @tf.function. If you are using AutoGraph, you can try decorating this function with @tf.function. If that does not work, then you may be using an unsupported feature or your source code may not be visible to AutoGraph. See https://github.com/tensorflow/tensorflow/blob/master/tensorflow/python/autograph/g3doc/reference/limitations.md#access-to-source-code for more information.


In [ ]:
train_dataset = create_time_series_window(train_dataset)
val_dataset = create_time_series_window(val_dataset)

TypeError: in user code:


    TypeError: outer_factory.<locals>.inner_factory.<locals>.<lambda>() takes 1 positional argument but 2 were given
